In [1]:
import os
from dotenv import load_dotenv
import time
import requests
import numpy as np

import pandas as pd
import anthropic
import re
import json
from math import ceil
load_dotenv()

True

In [2]:
# =============================
# PENGATURAN API CLAUDE (ANTHROPIC)
# =============================
# Pastikan ANTHROPIC_API_KEY sudah diset di environment variables atau file .env
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")

if not ANTHROPIC_API_KEY:
    raise EnvironmentError("ANTHROPIC_API_KEY belum dikonfigurasi di file .env")

url = "https://api.anthropic.com/v1/messages"

In [3]:

client = anthropic.Anthropic()

# message = client.messages.create(
#     model="claude-haiku-4-5-20251001",
#     max_tokens=1000,
#     messages=[
#         {
#             "role": "user",
#             "content": "What should I search for to find the latest developments in renewable energy?",
#         }
#     ],
# )
#print(message.content)

In [4]:
# =============================
# CATEGORY DEFINITIONS (New Version)
# =============================
CATEGORY_DEFINITIONS = {
    "lokasi": "Keluar karena pindah lokasi geografis secara eksplisit (pindah kota, luar kota, luar negeri/relokasi).",

    "perbedaan_musim_kehidupan": "Keluar karena perubahan fase hidup jangka panjang (menikah, punya anak, jenjang studi baru, perubahan karier besar). Bukan konflik jadwal rutin.",

    "tidak_ada_respon": "Tidak atau minim respons saat dihubungi, Missing in Action (tidak balas, sulit dihubungi, tidak pernah hadir).",

    "tertanam_di_gereja_lain": "Memilih tetap tertanam atau aktif di gereja lain, bukan di JPCC.",

    "waktu_tidak_sesuai": "Benturan jadwal atau komitmen waktu rutin (jam kerja, shift, pulang malam, jadwal kuliah).",

    "alasan_DATE": "Masalah atau kondisi terkait kelompok DATE (tidak cocok, beda usia, pindah DATE, konflik leader/member, DATE close/bubar).",

    "Admin": "Kesalahan administratif atau perubahan data yang bukan keputusan pribadi anggota (human error, new comer, probation, false positive).",

    "others": "Alasan tidak jelas, wafat, terlalu singkat, ambigu, atau tidak termasuk kategori lain."
}   

In [5]:
def classify_batch_with_claude_haiku(batch_texts):
    # Constructing the prompt using your exact zero-shot design
    system_prompt = f"""You are a data annotation system. Categories: {json.dumps(CATEGORY_DEFINITIONS, ensure_ascii=False)}
Rules:
- Choose EXACTLY ONE category per text.
- Label MUST match one of the category keys.
- If relocation is explicitly mentioned → choose "lokasi".
- If routine schedule conflict → choose "waktu_tidak_sesuai".
- If long-term life phase change → choose "perbedaan_musim_kehidupan".
- If related to DATE group condition/conflict → choose "alasan_DATE".
- If unclear or insufficient information → choose "others".
- Do not infer beyond the text.

Return a JSON array (same order as input).
Each object must contain:
- final_label (string)
- confidence (0–100 integer)
- keywords (Max 3 short keywords)

Output ONLY valid JSON. No explanations.
"""
# Combine the texts into a numbered format so Claude can easily track the count
    user_content = f"Here is a list of {len(batch_texts)} texts that need to be analyzed:\n"
    for idx, text in enumerate(batch_texts):
        user_content += f"[{idx+1}] {text}\n"

    try:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001", 
            max_tokens=8192,  # A large token limit to prevent JSON truncation
            temperature=0,    # 0 for deterministic zero-shot results
            system=system_prompt,
            messages=[
                {"role": "user", "content": user_content},
                # Assistant prefill to force Claude to start its response with a JSON Array
                {"role": "assistant", "content": "["} 
            ]
        )

        # 1. Ekstraksi teks asli dari Claude
        claude_text = response.content[ 0 ].text
        
        # 2. PEMBERSIHAN EKSTRA: Hapus kata '```json' dan '```'
        claude_text = claude_text.replace("```json", "").replace("```", "").strip()
        
        # 3. Gabungkan dengan kurung siku awalan
        raw_json = "[" + claude_text
        raw_json = raw_json.strip()
        
        # 4. PEMBERSIHAN EKSTRA: Pastikan tertutup dengan kurung siku
        if not raw_json.endswith("]"):
            raw_json += "]"
            
        # 5. PEMBERSIHAN EKSTRA: Hapus koma berlebih sebelum kurung siku tutup menggunakan Regex
        raw_json = re.sub(r',\s*\]$', ']', raw_json)

        outputs = json.loads(raw_json)
        return outputs
    
    except json.JSONDecodeError as e:
        # Jika masih gagal, ini akan mencetak teks aslinya ke terminal Anda!
        print("\n====== JSON ERROR DEBUGER ======")
        print(f"Gagal membaca JSON: {e}")
        print("Teks Mentah dari Claude:")
        print(raw_json)
        print("================================\n")
        raise Exception("Model did not return valid JSON")
    except Exception as e:
        raise Exception(f"API Error: {e}")

In [6]:
# split data df menjadi 5 file variable yang berbeda
df = pd.read_csv("freetext.csv")
df_split = np.array_split(df, 5)

for i, split_df in enumerate(df_split):
    split_df.to_csv(f"freetext_split_{i+1}.csv", index=False) 


c:\Users\Jovan\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [11]:
# =============================
# PIPELINE (BATCHED → STRUCTURED CSV)
# =============================

def run_pipeline(batch_size: int = 10):
    # Replace with your actual file name
    df = pd.read_csv("freetext_split_1.csv")

    results = []

    num_batches = ceil(len(df) / batch_size)

    for batch_idx in range(num_batches):
        start = batch_idx * batch_size
        end = min(start + batch_size, len(df))
        batch = df.iloc[start:end]

        batch_label_base = f"batch : {batch_idx + 1} of {num_batches}"
        print(f"Processing {batch_label_base}")

        try:
            outputs = classify_batch_with_claude_haiku(
                batch["delete_reason"].tolist()
            )

            # Protection to prevent output length mismatch
            if len(outputs) != len(batch):
                raise ValueError(f"Output length mismatch: Expected {len(batch)}, got {len(outputs)}")

        except Exception as e:
            print("Batch error:", e)
            # Fills with empty objects if the batch fails, matching your GPT pipeline behavior
            outputs = [{} for _ in range(len(batch))]

        for i, (_, row) in enumerate(batch.iterrows()):
            out = outputs[i] if i < len(outputs) else {}
            results.append({
                "delete_reason": row["delete_reason"],
                "batch": f"{batch_label_base} row: {i + 1}",
                "Final label": out.get("final_label", ""),
                "Confidence": f"{out.get('confidence', '')}%" if out.get("confidence") else "",
                "Keywords": ", ".join(out.get("keywords", [])) if out.get("keywords") else "",
            })

    # Save the results with a specific file name for Claude
    pd.DataFrame(results).to_csv("labeled_results_claude.csv", index=False)
    print("Saved to labeled_results_claude.csv")

if __name__ == "__main__":
    # Recommended batch size of 10 to ensure stability
    run_pipeline(batch_size=10) 

Processing batch : 1 of 48
Processing batch : 2 of 48
Processing batch : 3 of 48
Processing batch : 4 of 48
Processing batch : 5 of 48
Processing batch : 6 of 48
Processing batch : 7 of 48
Processing batch : 8 of 48
Processing batch : 9 of 48
Processing batch : 10 of 48
Processing batch : 11 of 48
Processing batch : 12 of 48
Processing batch : 13 of 48
Processing batch : 14 of 48
Processing batch : 15 of 48
Processing batch : 16 of 48
Processing batch : 17 of 48
Processing batch : 18 of 48
Processing batch : 19 of 48
Processing batch : 20 of 48
Processing batch : 21 of 48
Processing batch : 22 of 48
Processing batch : 23 of 48
Processing batch : 24 of 48
Processing batch : 25 of 48
Processing batch : 26 of 48
Processing batch : 27 of 48
Processing batch : 28 of 48
Processing batch : 29 of 48
Processing batch : 30 of 48
Processing batch : 31 of 48
Processing batch : 32 of 48
Processing batch : 33 of 48
Processing batch : 34 of 48
Processing batch : 35 of 48
Processing batch : 36 of 48
P

In [12]:
# =============================
# PIPELINE (BATCHED → STRUCTURED CSV)
# =============================

def run_pipeline(batch_size: int = 10):
    # Replace with your actual file name
    df = pd.read_csv("freetext_split_2.csv")

    results = []

    num_batches = ceil(len(df) / batch_size)

    for batch_idx in range(num_batches):
        start = batch_idx * batch_size
        end = min(start + batch_size, len(df))
        batch = df.iloc[start:end]

        batch_label_base = f"batch : {batch_idx + 1} of {num_batches}"
        print(f"Processing {batch_label_base}")

        try:
            outputs = classify_batch_with_claude_haiku(
                batch["delete_reason"].tolist()
            )

            # Protection to prevent output length mismatch
            if len(outputs) != len(batch):
                raise ValueError(f"Output length mismatch: Expected {len(batch)}, got {len(outputs)}")

        except Exception as e:
            print("Batch error:", e)
            # Fills with empty objects if the batch fails, matching your GPT pipeline behavior
            outputs = [{} for _ in range(len(batch))]

        for i, (_, row) in enumerate(batch.iterrows()):
            out = outputs[i] if i < len(outputs) else {}
            results.append({
                "delete_reason": row["delete_reason"],
                "batch": f"{batch_label_base} row: {i + 1}",
                "Final label": out.get("final_label", ""),
                "Confidence": f"{out.get('confidence', '')}%" if out.get("confidence") else "",
                "Keywords": ", ".join(out.get("keywords", [])) if out.get("keywords") else "",
            })

    # Save the results with a specific file name for Claude
    pd.DataFrame(results).to_csv("labeled_results_claude2.csv", index=False)
    print("Saved to labeled_results_claude2.csv")

if __name__ == "__main__":
    # Recommended batch size of 10 to ensure stability
    run_pipeline(batch_size=10) 

Processing batch : 1 of 48
Processing batch : 2 of 48
Processing batch : 3 of 48
Processing batch : 4 of 48
Processing batch : 5 of 48
Processing batch : 6 of 48
Processing batch : 7 of 48
Processing batch : 8 of 48
Processing batch : 9 of 48
Processing batch : 10 of 48
Processing batch : 11 of 48
Processing batch : 12 of 48
Processing batch : 13 of 48
Processing batch : 14 of 48
Processing batch : 15 of 48
Processing batch : 16 of 48
Processing batch : 17 of 48
Processing batch : 18 of 48
Processing batch : 19 of 48
Processing batch : 20 of 48
Processing batch : 21 of 48
Processing batch : 22 of 48
Processing batch : 23 of 48
Processing batch : 24 of 48
Processing batch : 25 of 48
Processing batch : 26 of 48
Processing batch : 27 of 48
Processing batch : 28 of 48
Processing batch : 29 of 48
Processing batch : 30 of 48
Processing batch : 31 of 48
Processing batch : 32 of 48
Processing batch : 33 of 48
Processing batch : 34 of 48
Processing batch : 35 of 48
Processing batch : 36 of 48
P

In [6]:
# =============================
# PIPELINE (BATCHED → STRUCTURED CSV)
# =============================

def run_pipeline(batch_size: int = 10):
    # Replace with your actual file name
    df = pd.read_csv("freetext_split_3.csv")

    results = []

    num_batches = ceil(len(df) / batch_size)

    for batch_idx in range(num_batches):
        start = batch_idx * batch_size
        end = min(start + batch_size, len(df))
        batch = df.iloc[start:end]

        batch_label_base = f"batch : {batch_idx + 1} of {num_batches}"
        print(f"Processing {batch_label_base}")

        try:
            outputs = classify_batch_with_claude_haiku(
                batch["delete_reason"].tolist()
            )

            # Protection to prevent output length mismatch
            if len(outputs) != len(batch):
                raise ValueError(f"Output length mismatch: Expected {len(batch)}, got {len(outputs)}")

        except Exception as e:
            print("Batch error:", e)
            # Fills with empty objects if the batch fails, matching your GPT pipeline behavior
            outputs = [{} for _ in range(len(batch))]

        for i, (_, row) in enumerate(batch.iterrows()):
            out = outputs[i] if i < len(outputs) else {}
            results.append({
                "delete_reason": row["delete_reason"],
                "batch": f"{batch_label_base} row: {i + 1}",
                "Final label": out.get("final_label", ""),
                "Confidence": f"{out.get('confidence', '')}%" if out.get("confidence") else "",
                "Keywords": ", ".join(out.get("keywords", [])) if out.get("keywords") else "",
            })

    # Save the results with a specific file name for Claude
    pd.DataFrame(results).to_csv("labeled_results_claude3.csv", index=False)
    print("Saved to labeled_results_claude3.csv")

if __name__ == "__main__":
    # Recommended batch size of 10 to ensure stability
    run_pipeline(batch_size=10) 

Processing batch : 1 of 48
Processing batch : 2 of 48
Processing batch : 3 of 48
Processing batch : 4 of 48
Processing batch : 5 of 48
Processing batch : 6 of 48
Processing batch : 7 of 48
Processing batch : 8 of 48
Processing batch : 9 of 48
Processing batch : 10 of 48
Processing batch : 11 of 48
Processing batch : 12 of 48
Processing batch : 13 of 48
Processing batch : 14 of 48
Processing batch : 15 of 48
Processing batch : 16 of 48
Processing batch : 17 of 48
Processing batch : 18 of 48
Processing batch : 19 of 48
Processing batch : 20 of 48
Processing batch : 21 of 48
Processing batch : 22 of 48
Processing batch : 23 of 48
Processing batch : 24 of 48
Processing batch : 25 of 48
Processing batch : 26 of 48
Processing batch : 27 of 48
Processing batch : 28 of 48
Processing batch : 29 of 48
Processing batch : 30 of 48
Processing batch : 31 of 48
Processing batch : 32 of 48
Processing batch : 33 of 48
Processing batch : 34 of 48
Processing batch : 35 of 48
Processing batch : 36 of 48
P

In [7]:
# =============================
# PIPELINE (BATCHED → STRUCTURED CSV)
# =============================

def run_pipeline(batch_size: int = 10):
    # Replace with your actual file name
    df = pd.read_csv("freetext_split_4.csv")

    results = []

    num_batches = ceil(len(df) / batch_size)

    for batch_idx in range(num_batches):
        start = batch_idx * batch_size
        end = min(start + batch_size, len(df))
        batch = df.iloc[start:end]

        batch_label_base = f"batch : {batch_idx + 1} of {num_batches}"
        print(f"Processing {batch_label_base}")

        try:
            outputs = classify_batch_with_claude_haiku(
                batch["delete_reason"].tolist()
            )

            # Protection to prevent output length mismatch
            if len(outputs) != len(batch):
                raise ValueError(f"Output length mismatch: Expected {len(batch)}, got {len(outputs)}")

        except Exception as e:
            print("Batch error:", e)
            # Fills with empty objects if the batch fails, matching your GPT pipeline behavior
            outputs = [{} for _ in range(len(batch))]

        for i, (_, row) in enumerate(batch.iterrows()):
            out = outputs[i] if i < len(outputs) else {}
            results.append({
                "delete_reason": row["delete_reason"],
                "batch": f"{batch_label_base} row: {i + 1}",
                "Final label": out.get("final_label", ""),
                "Confidence": f"{out.get('confidence', '')}%" if out.get("confidence") else "",
                "Keywords": ", ".join(out.get("keywords", [])) if out.get("keywords") else "",
            })

    # Save the results with a specific file name for Claude
    pd.DataFrame(results).to_csv("labeled_results_claude4.csv", index=False)
    print("Saved to labeled_results_claude4.csv")

if __name__ == "__main__":
    # Recommended batch size of 10 to ensure stability
    run_pipeline(batch_size=10) 

Processing batch : 1 of 48
Processing batch : 2 of 48
Processing batch : 3 of 48
Processing batch : 4 of 48
Processing batch : 5 of 48
Processing batch : 6 of 48
Processing batch : 7 of 48
Processing batch : 8 of 48
Processing batch : 9 of 48
Processing batch : 10 of 48
Processing batch : 11 of 48
Processing batch : 12 of 48
Processing batch : 13 of 48
Processing batch : 14 of 48
Processing batch : 15 of 48
Processing batch : 16 of 48
Processing batch : 17 of 48
Processing batch : 18 of 48
Processing batch : 19 of 48
Processing batch : 20 of 48
Processing batch : 21 of 48
Processing batch : 22 of 48
Processing batch : 23 of 48
Processing batch : 24 of 48
Processing batch : 25 of 48
Processing batch : 26 of 48
Processing batch : 27 of 48
Processing batch : 28 of 48
Processing batch : 29 of 48
Processing batch : 30 of 48
Processing batch : 31 of 48
Processing batch : 32 of 48
Processing batch : 33 of 48
Processing batch : 34 of 48
Processing batch : 35 of 48
Processing batch : 36 of 48
P

In [8]:
# =============================
# PIPELINE (BATCHED → STRUCTURED CSV)
# =============================

def run_pipeline(batch_size: int = 10):
    # Replace with your actual file name
    df = pd.read_csv("freetext_split_5.csv")

    results = []

    num_batches = ceil(len(df) / batch_size)

    for batch_idx in range(num_batches):
        start = batch_idx * batch_size
        end = min(start + batch_size, len(df))
        batch = df.iloc[start:end]

        batch_label_base = f"batch : {batch_idx + 1} of {num_batches}"
        print(f"Processing {batch_label_base}")

        try:
            outputs = classify_batch_with_claude_haiku(
                batch["delete_reason"].tolist()
            )

            # Protection to prevent output length mismatch
            if len(outputs) != len(batch):
                raise ValueError(f"Output length mismatch: Expected {len(batch)}, got {len(outputs)}")

        except Exception as e:
            print("Batch error:", e)
            # Fills with empty objects if the batch fails, matching your GPT pipeline behavior
            outputs = [{} for _ in range(len(batch))]

        for i, (_, row) in enumerate(batch.iterrows()):
            out = outputs[i] if i < len(outputs) else {}
            results.append({
                "delete_reason": row["delete_reason"],
                "batch": f"{batch_label_base} row: {i + 1}",
                "Final label": out.get("final_label", ""),
                "Confidence": f"{out.get('confidence', '')}%" if out.get("confidence") else "",
                "Keywords": ", ".join(out.get("keywords", [])) if out.get("keywords") else "",
            })

    # Save the results with a specific file name for Claude
    pd.DataFrame(results).to_csv("labeled_results_claude5.csv", index=False)
    print("Saved to labeled_results_claude5.csv")

if __name__ == "__main__":
    # Recommended batch size of 10 to ensure stability
    run_pipeline(batch_size=10) 

Processing batch : 1 of 48
Processing batch : 2 of 48
Processing batch : 3 of 48
Processing batch : 4 of 48
Processing batch : 5 of 48
Processing batch : 6 of 48
Processing batch : 7 of 48
Processing batch : 8 of 48
Processing batch : 9 of 48
Processing batch : 10 of 48
Processing batch : 11 of 48
Processing batch : 12 of 48
Processing batch : 13 of 48
Processing batch : 14 of 48
Processing batch : 15 of 48
Processing batch : 16 of 48
Processing batch : 17 of 48
Processing batch : 18 of 48
Processing batch : 19 of 48
Processing batch : 20 of 48
Processing batch : 21 of 48
Processing batch : 22 of 48
Processing batch : 23 of 48
Processing batch : 24 of 48
Processing batch : 25 of 48
Processing batch : 26 of 48
Processing batch : 27 of 48
Processing batch : 28 of 48
Processing batch : 29 of 48
Processing batch : 30 of 48
Processing batch : 31 of 48
Processing batch : 32 of 48
Processing batch : 33 of 48
Processing batch : 34 of 48
Processing batch : 35 of 48
Processing batch : 36 of 48
P

In [7]:
# =============================
# PIPELINE (BATCHED → STRUCTURED CSV)
# =============================

def run_pipeline(batch_size: int = 10):
    # Replace with your actual file name
    df = pd.read_csv("consistencydata.csv")

    results = []

    num_batches = ceil(len(df) / batch_size)

    for batch_idx in range(num_batches):
        start = batch_idx * batch_size
        end = min(start + batch_size, len(df))
        batch = df.iloc[start:end]

        batch_label_base = f"batch : {batch_idx + 1} of {num_batches}"
        print(f"Processing {batch_label_base}")

        try:
            outputs = classify_batch_with_claude_haiku(
                batch["delete_reason"].tolist()
            )

            # Protection to prevent output length mismatch
            if len(outputs) != len(batch):
                raise ValueError(f"Output length mismatch: Expected {len(batch)}, got {len(outputs)}")

        except Exception as e:
            print("Batch error:", e)
            # Fills with empty objects if the batch fails, matching your GPT pipeline behavior
            outputs = [{} for _ in range(len(batch))]

        for i, (_, row) in enumerate(batch.iterrows()):
            out = outputs[i] if i < len(outputs) else {}
            results.append({
                "delete_reason": row["delete_reason"],
                "batch": f"{batch_label_base} row: {i + 1}",
                "Final label": out.get("final_label", ""),
                "Confidence": f"{out.get('confidence', '')}%" if out.get("confidence") else "",
                "Keywords": ", ".join(out.get("keywords", [])) if out.get("keywords") else "",
            })

    # Save the results with a specific file name for Claude
    pd.DataFrame(results).to_csv("consistencyHaiku.csv", index=False)
    print("Saved to consistencyHaiku.csv")

if __name__ == "__main__":
    # Recommended batch size of 10 to ensure stability
    run_pipeline(batch_size=10) 

Processing batch : 1 of 25
Processing batch : 2 of 25
Processing batch : 3 of 25
Processing batch : 4 of 25
Processing batch : 5 of 25
Processing batch : 6 of 25
Processing batch : 7 of 25
Processing batch : 8 of 25
Processing batch : 9 of 25
Processing batch : 10 of 25
Processing batch : 11 of 25
Processing batch : 12 of 25
Processing batch : 13 of 25
Processing batch : 14 of 25
Processing batch : 15 of 25
Processing batch : 16 of 25
Processing batch : 17 of 25
Processing batch : 18 of 25
Processing batch : 19 of 25
Processing batch : 20 of 25
Processing batch : 21 of 25
Processing batch : 22 of 25
Processing batch : 23 of 25
Processing batch : 24 of 25
Processing batch : 25 of 25
Saved to consistencyHaiku.csv
